# 01 · Extract → Bronze → Clean → Silver

**Pipeline step 1 of the Weather End-to-End project**

| Layer | Table | What it holds |
|---|---|---|
| Bronze | `bronze.weather_raw` | Raw API data, exactly as received |
| Silver | `silver.weather_daily` | Cleaned, typed, enriched data |

**Before running:** start SQL Server with `docker compose up -d sqlserver` and wait ~30 seconds.

Data source: [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) — free, no API key needed.

In [2]:
import time
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option("display.max_columns", 30)

## 1. Database connection
Same settings as `docker-compose.yml`. If you changed the password there, change it here too.

In [4]:
DB_SERVER   = "localhost"
DB_PORT     = 1433
DB_USER     = "sa"
DB_PASSWORD = "Weather_Pass123!"
DB_NAME     = "WeatherDB"

def get_engine(database="master"):
    url = URL.create("mssql+pymssql", username=DB_USER, password=DB_PASSWORD,
                     host=DB_SERVER, port=DB_PORT, database=database)
    return create_engine(url)

# quick connection test
with get_engine().connect() as conn:
    print(conn.execute(text("SELECT @@VERSION")).scalar()[:60])

Microsoft SQL Server 2022 (RTM-CU27-GDR) (KB5126934) - 16.0.


## 2. Create the database + bronze / silver / gold schemas
Runs `sql/01_setup_database.sql`. `CREATE DATABASE` needs autocommit mode.

In [5]:
def run_sql_file(path, engine, autocommit=False):
    """Run a .sql file that uses GO as batch separator."""
    sql = Path(path).read_text(encoding="utf-8")
    batches = [b.strip() for b in sql.split("\nGO") if b.strip()]
    conn = engine.connect()
    if autocommit:
        conn = conn.execution_options(isolation_level="AUTOCOMMIT")
    with conn:
        for batch in batches:
            if batch.upper().startswith("USE "):      # we connect to the right DB instead
                continue
            conn.execute(text(batch))
        if not autocommit:
            conn.commit()

master = get_engine("master")
with master.connect().execution_options(isolation_level="AUTOCOMMIT") as conn:
    conn.execute(text("IF DB_ID('WeatherDB') IS NULL CREATE DATABASE WeatherDB"))

engine = get_engine(DB_NAME)
run_sql_file("../sql/01_setup_database.sql", engine, autocommit=True)

with engine.connect() as conn:
    print(pd.read_sql("SELECT name FROM sys.schemas WHERE name IN ('bronze','silver','gold')", conn))

     name
0  bronze
1    gold
2  silver


## 3. Extract — get weather data from the API
Daily data for 5 cities, from 2022-01-01 until about a week ago (the archive has a few days' delay).

In [6]:
CITIES = {
    "Baku":     {"country": "Azerbaijan", "lat": 40.41, "lon": 49.87},
    "London":   {"country": "UK",         "lat": 51.51, "lon": -0.13},
    "Berlin":   {"country": "Germany",    "lat": 52.52, "lon": 13.41},
    "New York": {"country": "USA",        "lat": 40.71, "lon": -74.01},
    "Dubai":    {"country": "UAE",        "lat": 25.20, "lon": 55.27},
}

# START_DATE = "2022-01-01"
# END_DATE   = (date.today() - timedelta(days=7)).isoformat()

ONE_DAY    = (date.today() - timedelta(days=1)).isoformat()   # yesterday
START_DATE = ONE_DAY
END_DATE   = ONE_DAY

DAILY_VARS = ["temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
              "precipitation_sum", "wind_speed_10m_max", "sunshine_duration"]

API_URL = "https://archive-api.open-meteo.com/v1/archive"


def fetch_city(city, info):
    params = {
        "latitude": info["lat"], "longitude": info["lon"],
        "start_date": START_DATE, "end_date": END_DATE,
        "daily": ",".join(DAILY_VARS), "timezone": "auto",
    }
    r = requests.get(API_URL, params=params, timeout=60)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["daily"])          # columns: time + the daily variables
    df["city"], df["country"] = city, info["country"]
    df["latitude"], df["longitude"] = info["lat"], info["lon"]
    return df


frames = []
for city, info in CITIES.items():
    print(f"Downloading {city} ...")
    frames.append(fetch_city(city, info))
    time.sleep(1)                                   # be polite to the free API

raw_df = pd.concat(frames, ignore_index=True)
print(raw_df.shape)
raw_df.head()

(5, 11)


,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,sunshine_duration,city,country,latitude,longitude
0,2026-10-16,25.1,20.4,22.4,0.0,18.6,38104.84,Baku,Azerbaijan,40.41,49.87
1,2026-10-16,15.7,8.9,13.3,5.4,15.9,25196.28,London,UK,51.51,-0.13
2,2026-10-16,17.8,6.1,11.5,0.0,9.3,35376.40,Berlin,Germany,52.52,13.41
3,2026-10-16,23.9,14.1,18.4,5.8,17.7,30829.84,New York,USA,40.71,-74.01
4,2026-10-16,35.3,27.4,31.6,0.0,13.0,38604.70,Dubai,UAE,25.20,55.27


REDA DATA FROM KAGLLE RO 

READ CSV DATA AND MERG WIHT API DATA 

## 4. Load → BRONZE (raw, unchanged + load metadata)

In [8]:
raw_df

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,sunshine_duration,city,country,latitude,longitude
0,2026-10-16,25.1,20.4,22.4,0.0,18.6,38104.84,Baku,Azerbaijan,40.41,49.87
1,2026-10-16,15.7,8.9,13.3,5.4,15.9,25196.28,London,UK,51.51,-0.13
2,2026-10-16,17.8,6.1,11.5,0.0,9.3,35376.40,Berlin,Germany,52.52,13.41
3,2026-10-16,23.9,14.1,18.4,5.8,17.7,30829.84,New York,USA,40.71,-74.01
4,2026-10-16,35.3,27.4,31.6,0.0,13.0,38604.70,Dubai,UAE,25.20,55.27


In [10]:
bronze_df = raw_df.copy()
bronze_df["source"] = "open-meteo-archive"
bronze_df["ingested_at"] = pd.Timestamp.now()

bronze_df.to_sql("weather_raw", engine, schema="bronze", if_exists="replace",
                 index=False, chunksize=100, method="multi")



5

In [11]:
with engine.connect() as conn:
    print("bronze rows:", conn.execute(text("SELECT COUNT(*) FROM WeatherDB.bronze.weather_raw")).scalar())

bronze rows: 5


In [14]:
bronze = pd.read_sql("SELECT * FROM bronze.weather_raw", engine)

In [15]:
bronze

,time,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,sunshine_duration,city,country,latitude,longitude,source,ingested_at
0,2026-10-16,25.1,20.4,22.4,0.0,18.6,38104.84,Baku,Azerbaijan,40.41,49.87,open-meteo-archive,2026-10-17 06:46:55.310
1,2026-10-16,15.7,8.9,13.3,5.4,15.9,25196.28,London,UK,51.51,-0.13,open-meteo-archive,2026-10-17 06:46:55.310
2,2026-10-16,17.8,6.1,11.5,0.0,9.3,35376.40,Berlin,Germany,52.52,13.41,open-meteo-archive,2026-10-17 06:46:55.310
3,2026-10-16,23.9,14.1,18.4,5.8,17.7,30829.84,New York,USA,40.71,-74.01,open-meteo-archive,2026-10-17 06:46:55.310
4,2026-10-16,35.3,27.4,31.6,0.0,13.0,38604.70,Dubai,UAE,25.20,55.27,open-meteo-archive,2026-10-17 06:46:55.310


## 5. Explore the raw data (read it back from bronze)

In [16]:
bronze = pd.read_sql("SELECT * FROM bronze.weather_raw", engine)

print("Data types:\n", bronze.dtypes, "\n")
print("Missing values per column:\n", bronze.isna().sum(), "\n")
print("Duplicate city+date rows:", bronze.duplicated(["city", "time"]).sum())
bronze.describe()

Data types:
 time                              str
temperature_2m_max            float64
temperature_2m_min            float64
temperature_2m_mean           float64
precipitation_sum             float64
wind_speed_10m_max            float64
sunshine_duration             float64
city                              str
country                           str
latitude                      float64
longitude                     float64
source                            str
ingested_at            datetime64[us]
dtype: object 

Missing values per column:
 time                   0
temperature_2m_max     0
temperature_2m_min     0
temperature_2m_mean    0
precipitation_sum      0
wind_speed_10m_max     0
sunshine_duration      0
city                   0
country                0
latitude               0
longitude              0
source                 0
ingested_at            0
dtype: int64 

Duplicate city+date rows: 0


,temperature_2m_max,temperature_2m_min,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,sunshine_duration,latitude,longitude,ingested_at
count,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5
mean,23.560000,15.380000,19.440000,2.240000,14.900000,33622.412000,42.070000,8.882000,2026-10-17 06:46:55.310000
min,15.700000,6.100000,11.500000,0.000000,9.300000,25196.280000,25.200000,-74.010000,2026-10-17 06:46:55.310000
25%,17.800000,8.900000,13.300000,0.000000,13.000000,30829.840000,40.410000,-0.130000,2026-10-17 06:46:55.310000
50%,23.900000,14.100000,18.400000,0.000000,15.900000,35376.400000,40.710000,13.410000,2026-10-17 06:46:55.310000
75%,25.100000,20.400000,22.400000,5.400000,17.700000,38104.840000,51.510000,49.870000,2026-10-17 06:46:55.310000
max,35.300000,27.400000,31.600000,5.800000,18.600000,38604.700000,52.520000,55.270000,2026-10-17 06:46:55.310000
std,7.669289,8.651994,8.038221,3.070505,3.791438,5630.053003,11.039907,51.972746,NaN


## 6. Clean → SILVER
Cleaning steps:
1. Rename columns to clear names
2. Fix data types (text date → real date)
3. Remove duplicates (same city + date)
4. Drop rows where *all* measurements are missing
5. Fill small gaps by interpolation (per city, in date order)
6. Remove impossible values (e.g. negative rain, temperature > 60 °C)
7. Convert sunshine seconds → hours
8. Add useful columns: `temp_range`, `year`, `month`, `season`, `is_rainy`

In [17]:
df = bronze.copy()

# 1. rename
df = df.rename(columns={
    "time": "date",
    "temperature_2m_max": "temp_max",
    "temperature_2m_min": "temp_min",
    "temperature_2m_mean": "temp_mean",
    "precipitation_sum": "precipitation_mm",
    "wind_speed_10m_max": "wind_speed_max_kmh",
    "sunshine_duration": "sunshine_sec",
})

# 2. types
df["date"] = pd.to_datetime(df["date"], errors="coerce")
measures = ["temp_max", "temp_min", "temp_mean", "precipitation_mm", "wind_speed_max_kmh", "sunshine_sec"]
df[measures] = df[measures].apply(pd.to_numeric, errors="coerce")
df["city"] = df["city"].str.strip()

# 3. duplicates + rows without a date
before = len(df)
df = df.dropna(subset=["date"]).drop_duplicates(subset=["city", "date"], keep="last")
print("removed duplicates / bad dates:", before - len(df))

# 4. rows with no measurements at all
before = len(df)
df = df.dropna(subset=measures, how="all")
print("removed empty rows:", before - len(df))

# 6. impossible values -> NaN (then filled in step 5)
df.loc[~df["temp_max"].between(-90, 60), "temp_max"] = np.nan
df.loc[~df["temp_min"].between(-90, 60), "temp_min"] = np.nan
df.loc[~df["temp_mean"].between(-90, 60), "temp_mean"] = np.nan
df.loc[df["precipitation_mm"] < 0, "precipitation_mm"] = np.nan
df.loc[df["wind_speed_max_kmh"] < 0, "wind_speed_max_kmh"] = np.nan
df.loc[~df["sunshine_sec"].between(0, 86400), "sunshine_sec"] = np.nan

# 5. fill gaps per city (linear in time), leftover edges with the city median
df = df.sort_values(["city", "date"]).reset_index(drop=True)
df[measures] = df.groupby("city")[measures].transform(lambda s: s.interpolate(limit_direction="both"))
df[measures] = df[measures].fillna(df.groupby("city")[measures].transform("median"))
df["precipitation_mm"] = df["precipitation_mm"].fillna(0)

# 7. sunshine to hours
df["sunshine_hours"] = (df["sunshine_sec"] / 3600).round(2)
df = df.drop(columns="sunshine_sec")

# 8. new columns
df["temp_range"] = (df["temp_max"] - df["temp_min"]).round(2)
df["year"]  = df["date"].dt.year
df["month"] = df["date"].dt.month
df["season"] = df["month"].map({12: "Winter", 1: "Winter", 2: "Winter",
                                3: "Spring", 4: "Spring", 5: "Spring",
                                6: "Summer", 7: "Summer", 8: "Summer",
                                9: "Autumn", 10: "Autumn", 11: "Autumn"})
df["is_rainy"] = (df["precipitation_mm"] >= 1.0).astype(int)
df["cleaned_at"] = pd.Timestamp.now()

silver_df = df[["city", "country", "latitude", "longitude", "date", "year", "month", "season",
                "temp_max", "temp_min", "temp_mean", "temp_range", "precipitation_mm",
                "wind_speed_max_kmh", "sunshine_hours", "is_rainy", "cleaned_at"]]
silver_df.head()

removed duplicates / bad dates: 0
removed empty rows: 0


,city,country,latitude,longitude,date,year,month,season,temp_max,temp_min,temp_mean,temp_range,precipitation_mm,wind_speed_max_kmh,sunshine_hours,is_rainy,cleaned_at
0,Baku,Azerbaijan,40.41,49.87,2026-10-16,2026,10,Autumn,25.1,20.4,22.4,4.7,0.0,18.6,10.58,0,2026-10-17 06:54:17.209687
1,Berlin,Germany,52.52,13.41,2026-10-16,2026,10,Autumn,17.8,6.1,11.5,11.7,0.0,9.3,9.83,0,2026-10-17 06:54:17.209687
2,Dubai,UAE,25.20,55.27,2026-10-16,2026,10,Autumn,35.3,27.4,31.6,7.9,0.0,13.0,10.72,0,2026-10-17 06:54:17.209687
3,London,UK,51.51,-0.13,2026-10-16,2026,10,Autumn,15.7,8.9,13.3,6.8,5.4,15.9,7.00,1,2026-10-17 06:54:17.209687
4,New York,USA,40.71,-74.01,2026-10-16,2026,10,Autumn,23.9,14.1,18.4,9.8,5.8,17.7,8.56,1,2026-10-17 06:54:17.209687


### Data-quality checks (the notebook stops here if something is wrong)

In [18]:
assert silver_df[["city", "date"]].duplicated().sum() == 0, "duplicates found"
assert silver_df.isna().sum().sum() == 0, "missing values left"
assert (silver_df["temp_min"] <= silver_df["temp_max"] + 0.01).mean() > 0.99, "min > max too often"
assert (silver_df["precipitation_mm"] >= 0).all(), "negative rain"
print("All checks passed ✔  rows:", len(silver_df))
silver_df.groupby("city").size()

All checks passed ✔  rows: 5


city
Baku        1
Berlin      1
Dubai       1
London      1
New York    1
dtype: int64

## 7. Load → SILVER

In [19]:
silver_df.to_sql("weather_daily", engine, schema="silver", if_exists="replace",
                 index=False, chunksize=100, method="multi")



5

In [20]:
with engine.connect() as conn:
    print(pd.read_sql("""
        SELECT city, COUNT(*) AS days, MIN([date]) AS first_day, MAX([date]) AS last_day,
               ROUND(AVG(temp_mean), 1) AS avg_temp
        FROM silver.weather_daily
        GROUP BY city
    """, conn))

       city  days  first_day   last_day  avg_temp
0      Baku     1 2026-10-16 2026-10-16      22.4
1    Berlin     1 2026-10-16 2026-10-16      11.5
2     Dubai     1 2026-10-16 2026-10-16      31.6
3    London     1 2026-10-16 2026-10-16      13.3
4  New York     1 2026-10-16 2026-10-16      18.4


✅ **Done.** Continue with `02_gold_ml_analysis.ipynb`.